# 02 - Fine-tuning LoRA no Google Colab

Notebook executavel em Colab limpo com GPU. Os checkpoints sao persistidos no Google Drive; o token e lido apenas de Secret/ambiente e nunca exibido.

## 1. Controles

Primeiro rode com `RUN_TRAINING = False`. Para preparar o `.venv` local, altere `INSTALL_LOCAL_DEPS = True` uma unica vez. Essa opcao reinstala PyTorch com CUDA se o kernel tiver PyTorch CPU-only. Depois que `torch.cuda.is_available()` retornar `True`, volte `INSTALL_LOCAL_DEPS = False` e altere `RUN_TRAINING = True`.

In [ ]:
# Colab bootstrap: run this cell first in a clean GPU runtime.
import os
import subprocess
from pathlib import Path

IN_COLAB = True
PROJECT_REPO_URL = "https://github.com/EngIaCeub/atelie-generativo-felipe-santiago.git"
COLAB_ROOT = Path("/content/atelie_generativo")
PROJECT_DIR = COLAB_ROOT / "repo"
subprocess.run(["nvidia-smi"], check=True)
subprocess.run(["git", "clone", PROJECT_REPO_URL, str(PROJECT_DIR)] if not PROJECT_DIR.exists() else ["git", "-C", str(PROJECT_DIR), "pull", "--ff-only"], check=True)
os.chdir(PROJECT_DIR)
try:
    from google.colab import drive, userdata
    drive.mount("/content/drive", force_remount=False)
    token = userdata.get("HF_TOKEN") or os.environ.get("HF_TOKEN")
    if token:
        os.environ["HF_TOKEN"] = token
        os.environ["HUGGING_FACE_HUB_TOKEN"] = token
except ImportError:
    raise RuntimeError("Este notebook deve ser executado no Google Colab.")
print("Projeto Colab:", PROJECT_DIR)


In [ ]:
# Colab bootstrap: run this cell first in a clean GPU runtime.
import os
import subprocess
from pathlib import Path

IN_COLAB = True
PROJECT_REPO_URL = "https://github.com/EngIaCeub/atelie-generativo-felipe-santiago.git"
COLAB_ROOT = Path("/content/atelie_generativo")
PROJECT_DIR = COLAB_ROOT / "repo"
subprocess.run(["nvidia-smi"], check=True)
subprocess.run(["git", "clone", PROJECT_REPO_URL, str(PROJECT_DIR)] if not PROJECT_DIR.exists() else ["git", "-C", str(PROJECT_DIR), "pull", "--ff-only"], check=True)
os.chdir(PROJECT_DIR)
try:
    from google.colab import drive, userdata
    drive.mount("/content/drive", force_remount=False)
    token = userdata.get("HF_TOKEN") or os.environ.get("HF_TOKEN")
    if token:
        os.environ["HF_TOKEN"] = token
        os.environ["HUGGING_FACE_HUB_TOKEN"] = token
except ImportError:
    raise RuntimeError("Este notebook deve ser executado no Google Colab.")
print("Projeto Colab:", PROJECT_DIR)


In [1]:
from __future__ import annotations

import csv
import datetime as dt
import hashlib
import importlib.util
import json
import os
import shutil
import subprocess
import sys
from getpass import getpass
from pathlib import Path

NOTEBOOK_REVISION = "colab-gpu-2026-07-11-v1"
RUN_TRAINING = False  # Mude para True para iniciar o treino real.
INSTALL_LOCAL_DEPS = False
ASK_HF_TOKEN_IF_MISSING = True
TORCH_CUDA_INDEX_URL = "https://download.pytorch.org/whl/cu128"
SELECTED_CONFIGS = ["config_a", "config_b"]
REQUIRED_LOCAL_PACKAGES = ["accelerate", "peft", "safetensors", "tensorboard", "datasets", "Pillow"]

ROOT = Path.cwd()
if not (ROOT / "config/project.json").exists() and (ROOT.parent / "config/project.json").exists():
    ROOT = ROOT.parent
CONFIG_PATH = ROOT / "config/project.json"
METADATA_PATH = ROOT / "dados/metadata.jsonl"
OUTPUT_ROOT = Path("/content/drive/MyDrive/atelie_generativo_felipe_santiago/treino_lora")
DIFFUSERS_DIR = Path("/content/atelie_generativo/diffusers")
DATASET_DIR = OUTPUT_ROOT / "_dataset_imagefolder"
EXPERIMENTS_CSV = OUTPUT_ROOT / "experimentos_colab.csv"
EXPECTED_VENV = Path("/content")

config = json.loads(CONFIG_PATH.read_text(encoding="utf-8"))
training_by_name = {item["name"]: item for item in config["training"]["configs"]}
missing = [name for name in SELECTED_CONFIGS if name not in training_by_name]
assert not missing, f"Config ausente em config/project.json: {missing}"

print("Notebook revision:", NOTEBOOK_REVISION)
print("Projeto:", config["project_name"])
print("Modelo base:", config["models"]["base_diffusion"])
print("Configs selecionadas:", SELECTED_CONFIGS)
print("RUN_TRAINING:", RUN_TRAINING)
print("INSTALL_LOCAL_DEPS:", INSTALL_LOCAL_DEPS)
print("Python:", sys.executable)
if EXPECTED_VENV.exists() and Path(sys.executable).resolve() != EXPECTED_VENV.resolve():
    print("AVISO: kernel atual nao e o .venv do projeto:", EXPECTED_VENV)

Notebook revision: local-nvidia-gpu-2026-07-11-v7
Projeto: atelie-generativo-felipe-santiago
Modelo base: stable-diffusion-v1-5/stable-diffusion-v1-5
Configs selecionadas: ['config_a', 'config_b']
RUN_TRAINING: True
INSTALL_LOCAL_DEPS: False
Python: c:\Users\felip\OneDrive\Documentos\Engenharia de Inteligência Artificial\Modelos Multimodais\sistematizacao\atelie-generativo-felipe-santiago\.venv\Scripts\python.exe


## 2. Funcoes de comando

`run()` mostra o comando antes de executar. Em dry-run, comandos `accelerate launch` sao apenas exibidos.

In [2]:
def run(cmd: list[str], cwd: Path | None = None, check: bool = True) -> subprocess.CompletedProcess[str]:
    printable = " ".join(str(part) for part in cmd)
    print("$", printable)
    first = Path(str(cmd[0])).name.lower() if cmd else ""
    if not RUN_TRAINING and first.startswith("accelerate") and len(cmd) > 1 and cmd[1] == "launch":
        print("DRY-RUN: comando de treino pesado nao executado.")
        return subprocess.CompletedProcess(cmd, 0, "", "")
    return subprocess.run([str(part) for part in cmd], cwd=str(cwd) if cwd else None, text=True, check=check)

def capture(cmd: list[str], cwd: Path | None = None) -> str:
    return subprocess.check_output([str(part) for part in cmd], cwd=str(cwd) if cwd else None, text=True).strip()

## 3. Ambiente NVIDIA, PyTorch CUDA e token

Esta celula verifica `nvidia-smi`, instala PyTorch CUDA se `INSTALL_LOCAL_DEPS = True`, e bloqueia treino real se CUDA nao estiver ativa no PyTorch. O valor de `HF_TOKEN` nao e exibido.

In [3]:
nvidia_smi = shutil.which("nvidia-smi")
if nvidia_smi:
    run([nvidia_smi])
else:
    print("nvidia-smi nao encontrado neste ambiente.")

if INSTALL_LOCAL_DEPS:
    run([sys.executable, "-m", "pip", "install", "--upgrade", "pip"])

try:
    import torch
except ModuleNotFoundError:
    torch = None

has_cuda_torch = bool(torch and torch.cuda.is_available())
torch_is_cpu_only = bool(torch and getattr(torch.version, "cuda", None) is None)
if INSTALL_LOCAL_DEPS and (torch is None or torch_is_cpu_only or not has_cuda_torch):
    print("Instalando PyTorch com CUDA no kernel atual...")
    run([
        sys.executable, "-m", "pip", "install", "--upgrade", "--force-reinstall",
        "torch", "torchvision", "--index-url", TORCH_CUDA_INDEX_URL,
    ])
    import importlib
    importlib.invalidate_caches()
    import torch
elif not INSTALL_LOCAL_DEPS:
    print("Instalacao adicional de PyTorch desativada (o Colab ja fornece CUDA); altere INSTALL_LOCAL_DEPS=True se precisar preparar o ambiente.")

if torch is None:
    if RUN_TRAINING:
        raise RuntimeError("PyTorch nao esta instalado neste ambiente Python.")
    print("PyTorch nao esta instalado; mantendo somente preparacao/dry-run.")
else:
    print("torch:", torch.__version__)
    print("torch.version.cuda:", torch.version.cuda)

cuda_ok = bool(torch and torch.cuda.is_available())
print("torch.cuda.is_available():", cuda_ok)
if cuda_ok:
    print("GPU PyTorch:", torch.cuda.get_device_name(0))
else:
    message = "CUDA indisponivel para PyTorch neste runtime."
    if RUN_TRAINING:
        raise RuntimeError(message + " Instale PyTorch CUDA no kernel atual antes de treinar.")
    print("AVISO:", message, "Treino pesado ficara em dry-run ate RUN_TRAINING=True em runtime com GPU.")

hf_token_available = bool(os.environ.get("HF_TOKEN"))
if not hf_token_available and ASK_HF_TOKEN_IF_MISSING:
    token = getpass("HF_TOKEN ausente. Cole o token ou pressione Enter para manter ausente: ").strip()
    if token:
        os.environ["HF_TOKEN"] = token
        os.environ["HUGGING_FACE_HUB_TOKEN"] = token
        hf_token_available = True
elif hf_token_available:
    os.environ["HUGGING_FACE_HUB_TOKEN"] = os.environ["HF_TOKEN"]
print("HF_TOKEN disponivel no ambiente:", hf_token_available)

$ C:\Windows\system32\nvidia-smi.EXE
Instalacao local de PyTorch CUDA desativada; altere INSTALL_LOCAL_DEPS=True se precisar preparar o ambiente.
torch: 2.11.0+cu128
torch.version.cuda: 12.8
torch.cuda.is_available(): True
GPU PyTorch: NVIDIA GeForce RTX 3050 Laptop GPU
HF_TOKEN disponivel no ambiente: True


## 4. Diffusers oficial e dependencias

Clona/atualiza o Diffusers oficial, instala os requisitos do exemplo no kernel atual, encontra `accelerate.exe` e registra o commit usado.

In [6]:
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

if INSTALL_LOCAL_DEPS:
    run([sys.executable, "-m", "pip", "install", "--upgrade", *REQUIRED_LOCAL_PACKAGES])
else:
    missing_packages = [pkg for pkg in REQUIRED_LOCAL_PACKAGES if importlib.util.find_spec("PIL" if pkg == "Pillow" else pkg) is None]
    if missing_packages:
        print("AVISO: pacotes ausentes no kernel atual:", missing_packages)
        print("Altere INSTALL_LOCAL_DEPS=True e rode novamente antes do treino real.")

if DIFFUSERS_DIR.exists():
    run(["git", "fetch", "--depth", "1", "origin", "main"], cwd=DIFFUSERS_DIR)
    run(["git", "checkout", "FETCH_HEAD"], cwd=DIFFUSERS_DIR)
else:
    run(["git", "clone", "--depth", "1", "--branch", "main", "https://github.com/huggingface/diffusers.git", DIFFUSERS_DIR])

DIFFUSERS_COMMIT = capture(["git", "rev-parse", "HEAD"], cwd=DIFFUSERS_DIR)
TRAIN_SCRIPT = DIFFUSERS_DIR / "examples/text_to_image/train_text_to_image_lora.py"
assert TRAIN_SCRIPT.exists(), f"Script oficial ausente: {TRAIN_SCRIPT}"

if INSTALL_LOCAL_DEPS:
    run([sys.executable, "-m", "pip", "install", "-e", DIFFUSERS_DIR])
    run([sys.executable, "-m", "pip", "install", "-r", DIFFUSERS_DIR / "examples/text_to_image/requirements.txt"])

accelerate_exe = shutil.which("accelerate")
if not accelerate_exe:
    accelerate_candidate = Path(sys.executable).with_name("accelerate.exe")
    if accelerate_candidate.exists():
        accelerate_exe = str(accelerate_candidate)
if not accelerate_exe:
    raise RuntimeError("Accelerate nao encontrado no PATH nem ao lado do Python do kernel. Rode com INSTALL_LOCAL_DEPS=True neste kernel.")

ACCELERATE_CONFIG = OUTPUT_ROOT / "accelerate_config.yaml"
ACCELERATE_CONFIG.parent.mkdir(parents=True, exist_ok=True)
accelerate_config_text = """compute_environment: LOCAL_MACHINE
debug: false
distributed_type: 'NO'
downcast_bf16: false
enable_cpu_affinity: false
machine_rank: 0
main_training_function: main
mixed_precision: fp16
num_machines: 1
num_processes: 1
rdzv_backend: static
same_network: false
tpu_use_cluster: false
tpu_use_sudo: false
use_cpu: false
"""
ACCELERATE_CONFIG.write_text(accelerate_config_text, encoding="utf-8")
ACCELERATE_CMD = [accelerate_exe]

print("Diffusers commit:", DIFFUSERS_COMMIT)
print("Script oficial:", TRAIN_SCRIPT)
print("Accelerate:", accelerate_exe)
print("Accelerate config:", ACCELERATE_CONFIG)
print(ACCELERATE_CONFIG.read_text(encoding="utf-8"))


$ git fetch --depth 1 origin main
$ git checkout FETCH_HEAD
Diffusers commit: 01969142b55379991fee07608c9e7e8f80afced0
Script oficial: C:\ag-diffusers\examples\text_to_image\train_text_to_image_lora.py
Accelerate: c:\Users\felip\OneDrive\Documentos\Engenharia de Inteligência Artificial\Modelos Multimodais\sistematizacao\atelie-generativo-felipe-santiago\.venv\Scripts\accelerate.EXE
Accelerate config: c:\Users\felip\OneDrive\Documentos\Engenharia de Inteligência Artificial\Modelos Multimodais\sistematizacao\atelie-generativo-felipe-santiago\resultados\treino\local\accelerate_config.yaml
compute_environment: LOCAL_MACHINE
debug: false
distributed_type: 'NO'
downcast_bf16: false
enable_cpu_affinity: false
machine_rank: 0
main_training_function: main
mixed_precision: fp16
num_machines: 1
num_processes: 1
rdzv_backend: static
same_network: false
tpu_use_cluster: false
tpu_use_sudo: false
use_cpu: false



## 5. Dataset validado e hash

Converte `dados/metadata.jsonl` para formato `imagefolder` temporario em `resultados/treino/local/_dataset_imagefolder` e calcula hash do dataset.

In [7]:
def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

def dataset_hash() -> str:
    digest = hashlib.sha256()
    paths = [METADATA_PATH]
    paths.extend(sorted((ROOT / "dados/imagens").glob("*.png")))
    for path in paths:
        digest.update(path.relative_to(ROOT).as_posix().encode("utf-8"))
        digest.update(b"\0")
        digest.update(sha256_file(path).encode("ascii"))
        digest.update(b"\0")
    return digest.hexdigest()

records = [json.loads(line) for line in METADATA_PATH.read_text(encoding="utf-8").splitlines() if line.strip()]
assert 20 <= len(records) <= 40, f"Quantidade invalida de registros: {len(records)}"

if DATASET_DIR.exists():
    shutil.rmtree(DATASET_DIR)
DATASET_DIR.mkdir(parents=True)

rewritten = []
trigger = config["style"]["trigger_token"]
for record in records:
    source = ROOT / record["file_name"]
    assert source.exists(), f"Imagem ausente: {source}"
    assert record["text"].startswith(trigger), f"Caption sem trigger token: {source.name}"
    shutil.copy2(source, DATASET_DIR / source.name)
    rewritten.append({"file_name": source.name, "text": record["text"]})

with (DATASET_DIR / "metadata.jsonl").open("w", encoding="utf-8") as handle:
    for record in rewritten:
        handle.write(json.dumps(record, ensure_ascii=False) + "\n")

DATASET_HASH = dataset_hash()
PROJECT_COMMIT = capture(["git", "rev-parse", "HEAD"], cwd=ROOT)
print("Registros:", len(records))
print("Dataset preparado:", DATASET_DIR)
print("dataset_hash:", DATASET_HASH)
print("project_commit:", PROJECT_COMMIT)


Registros: 24
Dataset preparado: c:\Users\felip\OneDrive\Documentos\Engenharia de Inteligência Artificial\Modelos Multimodais\sistematizacao\atelie-generativo-felipe-santiago\resultados\treino\local\_dataset_imagefolder
dataset_hash: 7e34d52d32ae06c83440d8dc176ea7ad472980b1380d1972c297d5ab6dabec31
project_commit: b3a817e9a6ab4f4d9ffb3705c2550be707812837


## 6. Funcoes de treino e registro

Cada configuracao registra inicio/fim no CSV. `push_to_hub` fica desativado ate aprovacao humana da melhor configuracao.

In [8]:
EXPERIMENT_FIELDS = [
    "experimento", "data", "commit", "diffusers_commit", "dataset_hash", "seed", "rank", "learning_rate",
    "max_train_steps", "status", "checkpoint", "hub_url", "observacoes",
]

def checkpoint_step(path: Path) -> int:
    try:
        return int(path.name.rsplit("-", 1)[1])
    except (IndexError, ValueError):
        return -1

def migrate_experiments_csv() -> None:
    if not EXPERIMENTS_CSV.exists():
        return
    old_fields = [
        "experimento", "data", "commit", "dataset_hash", "seed", "rank", "learning_rate",
        "max_train_steps", "status", "checkpoint", "hub_url", "observacoes",
    ]
    with EXPERIMENTS_CSV.open(newline="", encoding="utf-8") as handle:
        rows = list(csv.reader(handle))
    if not rows or rows[0] == EXPERIMENT_FIELDS:
        return

    migrated = []
    for values in rows[1:]:
        if not values:
            continue
        if len(values) == len(EXPERIMENT_FIELDS):
            row = dict(zip(EXPERIMENT_FIELDS, values))
        else:
            row = dict(zip(old_fields, values))
            notes = row.get("observacoes", "")
            marker = "diffusers_commit="
            if marker in notes:
                row["diffusers_commit"] = notes.split(marker, 1)[1].split(";", 1)[0].split()[0].strip(".")
            else:
                row["diffusers_commit"] = ""
        migrated.append({field: row.get(field, "") for field in EXPERIMENT_FIELDS})

    with EXPERIMENTS_CSV.open("w", newline="", encoding="utf-8") as handle:
        writer = csv.DictWriter(handle, fieldnames=EXPERIMENT_FIELDS)
        writer.writeheader()
        writer.writerows(migrated)

def append_experiment(row: dict[str, object]) -> None:
    EXPERIMENTS_CSV.parent.mkdir(parents=True, exist_ok=True)
    migrate_experiments_csv()
    exists = EXPERIMENTS_CSV.exists()
    with EXPERIMENTS_CSV.open("a", newline="", encoding="utf-8") as handle:
        writer = csv.DictWriter(handle, fieldnames=EXPERIMENT_FIELDS)
        if not exists:
            writer.writeheader()
        writer.writerow({field: row.get(field, "") for field in EXPERIMENT_FIELDS})

def latest_checkpoint(output_dir: Path) -> str:
    checkpoints = sorted(output_dir.glob("checkpoint-*"), key=checkpoint_step)
    return str(checkpoints[-1]) if checkpoints else str(output_dir)

def ensure_training_context() -> None:
    global PROJECT_COMMIT, DATASET_HASH

    missing_deps = [
        name for name in ("ACCELERATE_CMD", "TRAIN_SCRIPT", "DIFFUSERS_COMMIT")
        if name not in globals()
    ]
    if missing_deps:
        raise RuntimeError(
            "Contexto de dependencias ausente: "
            + ", ".join(missing_deps)
            + ". Rode a celula 'Diffusers oficial e dependencias' antes do treino."
        )

    if "PROJECT_COMMIT" not in globals():
        PROJECT_COMMIT = capture(["git", "rev-parse", "HEAD"], cwd=ROOT)
        print("project_commit recalculado:", PROJECT_COMMIT)

    if "DATASET_HASH" in globals() and DATASET_DIR.exists() and (DATASET_DIR / "metadata.jsonl").exists():
        return

    print("DATASET_HASH ausente no kernel atual; preparando dataset antes do treino...")

    def sha256_file_local(path: Path) -> str:
        digest = hashlib.sha256()
        with path.open("rb") as handle:
            for chunk in iter(lambda: handle.read(1024 * 1024), b""):
                digest.update(chunk)
        return digest.hexdigest()

    records = [json.loads(line) for line in METADATA_PATH.read_text(encoding="utf-8").splitlines() if line.strip()]
    assert 20 <= len(records) <= 40, f"Quantidade invalida de registros: {len(records)}"

    if DATASET_DIR.exists():
        shutil.rmtree(DATASET_DIR)
    DATASET_DIR.mkdir(parents=True)

    trigger = config["style"]["trigger_token"]
    rewritten = []
    for record in records:
        source = ROOT / record["file_name"]
        assert source.exists(), f"Imagem ausente: {source}"
        assert record["text"].startswith(trigger), f"Caption sem trigger token: {source.name}"
        shutil.copy2(source, DATASET_DIR / source.name)
        rewritten.append({"file_name": source.name, "text": record["text"]})

    with (DATASET_DIR / "metadata.jsonl").open("w", encoding="utf-8") as handle:
        for record in rewritten:
            handle.write(json.dumps(record, ensure_ascii=False) + "\n")

    digest = hashlib.sha256()
    paths = [METADATA_PATH]
    paths.extend(sorted((ROOT / "dados/imagens").glob("*.png")))
    for path in paths:
        digest.update(path.relative_to(ROOT).as_posix().encode("utf-8"))
        digest.update(b"\0")
        digest.update(sha256_file_local(path).encode("ascii"))
        digest.update(b"\0")
    DATASET_HASH = digest.hexdigest()
    print("Registros:", len(records))
    print("Dataset preparado:", DATASET_DIR)
    print("dataset_hash recalculado:", DATASET_HASH)

def run_lora_config(cfg: dict) -> None:
    ensure_training_context()

    name = cfg["name"]
    output_dir = OUTPUT_ROOT / name
    output_dir.mkdir(parents=True, exist_ok=True)
    common = {
        "experimento": name,
        "commit": PROJECT_COMMIT,
        "diffusers_commit": DIFFUSERS_COMMIT,
        "dataset_hash": DATASET_HASH,
        "seed": config["training"]["seed"],
        "rank": cfg["rank"],
        "learning_rate": cfg["learning_rate"],
        "max_train_steps": cfg["max_train_steps"],
        "hub_url": "",
    }
    append_experiment({
        **common,
        "data": dt.datetime.now(dt.timezone.utc).isoformat(),
        "status": "started" if RUN_TRAINING else "planned_dry_run",
        "checkpoint": "",
        "observacoes": "LoRA local NVIDIA; resume_from_checkpoint=latest; push_to_hub desativado.",
    })

    cmd = [
        *ACCELERATE_CMD, "launch", "--config_file", str(ACCELERATE_CONFIG), str(TRAIN_SCRIPT),
        "--pretrained_model_name_or_path", config["models"]["base_diffusion"],
        "--train_data_dir", str(DATASET_DIR),
        "--image_column", "image",
        "--caption_column", "text",
        "--resolution", str(config["training"]["resolution"]),
        "--center_crop",
        "--random_flip",
        "--train_batch_size", "1",
        "--gradient_accumulation_steps", "4",
        "--gradient_checkpointing",
        "--max_train_steps", str(cfg["max_train_steps"]),
        "--learning_rate", str(cfg["learning_rate"]),
        "--lr_scheduler", "constant",
        "--lr_warmup_steps", "0",
        "--rank", str(cfg["rank"]),
        "--checkpointing_steps", "250",
        "--checkpoints_total_limit", "3",
        "--resume_from_checkpoint", "latest",
        "--seed", str(config["training"]["seed"]),
        "--mixed_precision", "fp16",
        "--dataloader_num_workers", "0",
        "--validation_prompt", f"{config['style']['trigger_token']}, um lobo-guara no Cerrado em xilogravura digital",
        "--num_validation_images", "2",
        "--validation_epochs", "1",
        "--output_dir", str(output_dir),
        "--report_to", "tensorboard",
    ]

    status = "completed_dry_run"
    checkpoint = ""
    notes = "Comando mostrado; treino pesado nao executado."
    try:
        run(cmd)
        if RUN_TRAINING:
            status = "concluido"
            checkpoint = latest_checkpoint(output_dir)
            notes = "Treino local concluido; revisar validacoes antes de publicar no Hub."
    except Exception as exc:
        status = "falhou"
        notes = f"Falha real: {type(exc).__name__}: {exc}"
        raise
    finally:
        append_experiment({
            **common,
            "data": dt.datetime.now(dt.timezone.utc).isoformat(),
            "status": status,
            "checkpoint": checkpoint,
            "observacoes": notes,
        })

## 7. Configuracao A

In [7]:
run_lora_config(training_by_name["config_a"])

$ c:\Users\felip\OneDrive\Documentos\Engenharia de Inteligência Artificial\Modelos Multimodais\sistematizacao\atelie-generativo-felipe-santiago\.venv\Scripts\accelerate.EXE launch --config_file c:\Users\felip\OneDrive\Documentos\Engenharia de Inteligência Artificial\Modelos Multimodais\sistematizacao\atelie-generativo-felipe-santiago\resultados\treino\local\accelerate_config.yaml C:\ag-diffusers\examples\text_to_image\train_text_to_image_lora.py --pretrained_model_name_or_path stable-diffusion-v1-5/stable-diffusion-v1-5 --train_data_dir c:\Users\felip\OneDrive\Documentos\Engenharia de Inteligência Artificial\Modelos Multimodais\sistematizacao\atelie-generativo-felipe-santiago\resultados\treino\local\_dataset_imagefolder --image_column image --caption_column text --resolution 512 --center_crop --random_flip --train_batch_size 1 --gradient_accumulation_steps 4 --gradient_checkpointing --max_train_steps 1000 --learning_rate 0.0001 --lr_scheduler constant --lr_warmup_steps 0 --rank 4 --che

## 8. Configuracao B

In [9]:
run_lora_config(training_by_name["config_b"])

$ c:\Users\felip\OneDrive\Documentos\Engenharia de Inteligência Artificial\Modelos Multimodais\sistematizacao\atelie-generativo-felipe-santiago\.venv\Scripts\accelerate.EXE launch --config_file c:\Users\felip\OneDrive\Documentos\Engenharia de Inteligência Artificial\Modelos Multimodais\sistematizacao\atelie-generativo-felipe-santiago\resultados\treino\local\accelerate_config.yaml C:\ag-diffusers\examples\text_to_image\train_text_to_image_lora.py --pretrained_model_name_or_path stable-diffusion-v1-5/stable-diffusion-v1-5 --train_data_dir c:\Users\felip\OneDrive\Documentos\Engenharia de Inteligência Artificial\Modelos Multimodais\sistematizacao\atelie-generativo-felipe-santiago\resultados\treino\local\_dataset_imagefolder --image_column image --caption_column text --resolution 512 --center_crop --random_flip --train_batch_size 1 --gradient_accumulation_steps 4 --gradient_checkpointing --max_train_steps 1500 --learning_rate 5e-05 --lr_scheduler constant --lr_warmup_steps 0 --rank 8 --chec

## 9. Validacao local apos execucao

Execute somente depois que as celulas anteriores terminarem. Se o treino pesado falhar, mantenha o erro real e nao preencha metricas manualmente.

In [11]:
os.environ.setdefault("PYTHONIOENCODING", "utf-8")
run([sys.executable, str(ROOT / "scripts/check_secrets.py")], cwd=ROOT)
run([sys.executable, str(ROOT / "scripts/project_status.py")], cwd=ROOT)

$ c:\Users\felip\OneDrive\Documentos\Engenharia de Inteligência Artificial\Modelos Multimodais\sistematizacao\atelie-generativo-felipe-santiago\.venv\Scripts\python.exe c:\Users\felip\OneDrive\Documentos\Engenharia de Inteligência Artificial\Modelos Multimodais\sistematizacao\atelie-generativo-felipe-santiago\scripts\check_secrets.py
$ c:\Users\felip\OneDrive\Documentos\Engenharia de Inteligência Artificial\Modelos Multimodais\sistematizacao\atelie-generativo-felipe-santiago\.venv\Scripts\python.exe c:\Users\felip\OneDrive\Documentos\Engenharia de Inteligência Artificial\Modelos Multimodais\sistematizacao\atelie-generativo-felipe-santiago\scripts\project_status.py


CompletedProcess(args=['c:\\Users\\felip\\OneDrive\\Documentos\\Engenharia de Inteligência Artificial\\Modelos Multimodais\\sistematizacao\\atelie-generativo-felipe-santiago\\.venv\\Scripts\\python.exe', 'c:\\Users\\felip\\OneDrive\\Documentos\\Engenharia de Inteligência Artificial\\Modelos Multimodais\\sistematizacao\\atelie-generativo-felipe-santiago\\scripts\\project_status.py'], returncode=0)

## Portao humano

A publicacao no Hugging Face fica bloqueada ate revisao humana da melhor configuracao. Depois da escolha, registrar justificativa, atualizar model card e publicar os pesos em uma etapa separada.